In [1]:
import findspark
findspark.init()

from pyspark.sql import SparkSession

# Configuração  Apache Iceberg
spark = SparkSession.builder \
    .appName("Trabalho_Iceberg_Exclusivo") \
    .config("spark.jars.packages", "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.5.0") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.local", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.local.type", "hadoop") \
    .config("spark.sql.catalog.local.warehouse", "warehouse_iceberg") \
    .getOrCreate()

print("Apache Iceberg configurado ")

26/04/28 15:25:15 WARN Utils: Your hostname, MSI resolves to a loopback address: 127.0.1.1; using 172.17.119.163 instead (on interface eth0)
26/04/28 15:25:15 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/elison/trabalho_eng_dados/.venv/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/elison/.ivy2/cache
The jars for the packages stored in: /home/elison/.ivy2/jars
org.apache.iceberg#iceberg-spark-runtime-3.5_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-e9ea8f88-6c72-4645-af8c-f93633b9d568;1.0
	confs: [default]
	found org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.5.0 in central
:: resolution report :: resolve 136ms :: artifacts dl 9ms
	:: modules in use:
	org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.5.0 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   1   |   0   |   0   |   0   ||   1   |   0   |
	---------------------------------------------------------------------
:: retrieving :: org.apache.sp

Apache Iceberg configurado 


In [2]:
spark.sql("CREATE DATABASE IF NOT EXISTS local.db_projeto")

DataFrame[]

In [3]:
spark.sql("""
    CREATE TABLE IF NOT EXISTS local.db_projeto.tabela_jogos (
        id bigint,
        nome string,
        genero string,
        ano int
    ) 
    USING iceberg
""")

DataFrame[]

In [4]:
spark.sql("""
    INSERT INTO local.db_projeto.tabela_jogos VALUES
    (1, 'FIFA 23', 'Esporte', 2023),
    (2, 'God of War', 'Ação', 2022),
    (3, 'Minecraft', 'Sandbox', 2011)
""")
print(" Dados inseridos ")

 Dados inseridos 


In [5]:
# Mudando o gênero do Minecraft para 'Construção'
spark.sql("""
    UPDATE local.db_projeto.tabela_jogos 
    SET genero = 'Construção' 
    WHERE id = 3
""")
print("Update realizado ")

Update realizado 


In [6]:
# Removendo o jogo FIFA (ID 1)
spark.sql("DELETE FROM local.db_projeto.tabela_jogos WHERE id = 1")
print(" Delete realizado")

 Delete realizado


In [7]:
spark.sql("SELECT * FROM local.db_projeto.tabela_jogos").show()

+---+----------+----------+----+
| id|      nome|    genero| ano|
+---+----------+----------+----+
|  3| Minecraft|Construção|2011|
|  2|God of War|      Ação|2022|
+---+----------+----------+----+



In [8]:
# Mostra o ID da versão, quando foi feita e que tipo de operação foi (append, delete, overwrite)
spark.sql("""
    SELECT committed_at, snapshot_id, operation 
    FROM local.db_projeto.tabela_jogos.snapshots 
    ORDER BY committed_at ASC
""").show(truncate=False)

+-----------------------+-------------------+---------+
|committed_at           |snapshot_id        |operation|
+-----------------------+-------------------+---------+
|2026-04-28 15:25:37.588|6618643536360176253|append   |
|2026-04-28 15:25:43.81 |6103581642117016878|overwrite|
|2026-04-28 15:25:47.112|8413611430483298030|delete   |
+-----------------------+-------------------+---------+



In [9]:
# Verificar histórico
spark.sql("SELECT * FROM local.db_projeto.tabela_jogos.history").show()

+--------------------+-------------------+-------------------+-------------------+
|     made_current_at|        snapshot_id|          parent_id|is_current_ancestor|
+--------------------+-------------------+-------------------+-------------------+
|2026-04-28 15:25:...|6618643536360176253|               NULL|               true|
|2026-04-28 15:25:...|6103581642117016878|6618643536360176253|               true|
|2026-04-28 15:25:...|8413611430483298030|6103581642117016878|               true|
+--------------------+-------------------+-------------------+-------------------+



In [ ]:
# Substituir o número abaixo por um snapshot_id do  histórico
id_antigo = 1551280330427088984
df_passado = spark.read \
    .option("snapshot-id", id_antigo) \
    .table("local.db_projeto.tabela_jogos")

print(" Lendo a tabela (com duplicadas):")
df_passado.show()